# UCF-Crime Class Description Prototype Generation

This notebook creates class-level visual-action prototypes from existing GPT video descriptions. It does not train VadCLIP and does not modify model code.

## 1. Config

Keep `DRY_RUN = True` to validate the full pipeline without calling OpenAI APIs. Set it to `False` after pasting the API key.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import math
import random
import re
import time
from collections import Counter, defaultdict

import numpy as np

try:
    import requests
except ImportError as exc:
    raise ImportError("Please install requests first, for example: pip install requests") from exc

# Use a small NumPy KMeans implementation below to avoid dependency/version issues.
KMeans = None

OPENAI_API_KEY = ""
EMBEDDING_MODEL = "text-embedding-3-small"
GENERATION_MODEL = "gpt-5.6-luna"

SPLITS_TO_USE = ["train"]
USE_VADCLIP_TRAIN_LIST = True
FINAL_PROTOTYPES_PER_CLASS = 5
CANDIDATES_PER_CLUSTER = 3
MAX_DESCRIPTIONS_PER_CLUSTER_PROMPT = 35
MIN_DESCRIPTION_WORDS = 10
RANDOM_SEED = 42

DRY_RUN = False
REQUEST_TIMEOUT_SECONDS = 180
SLEEP_BETWEEN_REQUESTS_SECONDS = 0.25
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 1.5
EMBEDDING_BATCH_SIZE = 128
GENERATION_MAX_OUTPUT_TOKENS = 900

current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / "code").exists() else current_dir.parent
CODE_DIR = PROJECT_ROOT / "code"
DESCRIPTION_JSON = CODE_DIR / "ucf_gpt_video_descriptions.json"
TRAIN_LIST_CSV = PROJECT_ROOT / "VadCLIP" / "list" / "ucf_CLIP_rgb_description.csv"
EMBEDDING_CACHE_JSON = CODE_DIR / "ucf_class_prototype_embeddings_cache.json"
OUTPUT_JSON = CODE_DIR / "ucf_class_description_prototypes.json"
OUTPUT_CSV = CODE_DIR / "ucf_class_description_prototypes.csv"
REPORT_JSON = CODE_DIR / "ucf_class_prototype_generation_report.json"

if USE_VADCLIP_TRAIN_LIST:
    EMBEDDING_CACHE_JSON = CODE_DIR / "ucf_class_prototype_embeddings_cache_vadclip_train.json"
    OUTPUT_JSON = CODE_DIR / "ucf_class_description_prototypes_vadclip_train.json"
    OUTPUT_CSV = CODE_DIR / "ucf_class_description_prototypes_vadclip_train.csv"
    REPORT_JSON = CODE_DIR / "ucf_class_prototype_generation_report_vadclip_train.json"

if DRY_RUN:
    dryrun_suffix = "vadclip_train_dryrun" if USE_VADCLIP_TRAIN_LIST else "dryrun"
    EMBEDDING_CACHE_JSON = CODE_DIR / f"ucf_class_prototype_embeddings_cache_{dryrun_suffix}.json"
    OUTPUT_JSON = CODE_DIR / f"ucf_class_description_prototypes_{dryrun_suffix}.json"
    OUTPUT_CSV = CODE_DIR / f"ucf_class_description_prototypes_{dryrun_suffix}.csv"
    REPORT_JSON = CODE_DIR / f"ucf_class_prototype_generation_report_{dryrun_suffix}.json"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Project root:", PROJECT_ROOT)
print("Description JSON:", DESCRIPTION_JSON)
print("Use VadCLIP train list:", USE_VADCLIP_TRAIN_LIST)
print("Train list CSV:", TRAIN_LIST_CSV)
print("Dry run:", DRY_RUN)

Project root: D:\Finetune VadCLIP
Description JSON: D:\Finetune VadCLIP\code\ucf_gpt_video_descriptions.json
Use VadCLIP train list: True
Train list CSV: D:\Finetune VadCLIP\VadCLIP\list\ucf_CLIP_rgb_description.csv
Dry run: False


## 2. Shared Helpers

In [2]:
FORBIDDEN_LABELS = [
    "abuse",
    "arrest",
    "arson",
    "assault",
    "burglary",
    "explosion",
    "fighting",
    "road accident",
    "roadaccident",
    "robbery",
    "shooting",
    "shoplifting",
    "stealing",
    "vandalism",
    "anomaly",
    "anomalous",
    "crime",
]

PROTOTYPE_TYPES = [
    "core_action",
    "actor_object_interaction",
    "temporal_progression",
    "scene_context",
    "hard_negative_distinction",
]


def api_key_is_filled(value, placeholder="PASTE_OPENAI_API_KEY_HERE"):
    return bool(value and value.strip() and value.strip() != placeholder)


def normalize_text_artifacts(text):
    if text is None:
        return ""
    text = str(text)
    replacements = {
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2013": "-",
        "\u2014": "-",
        "\u2026": "...",
    }
    for src, dst in replacements.items():
        text = text.replace(src, dst)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def count_words(text):
    return len(re.findall(r"\b[\w'-]+\b", text or ""))


def find_forbidden_labels(text):
    text_norm = (text or "").lower()
    found = []
    for label in FORBIDDEN_LABELS:
        pattern = r"\b" + re.escape(label).replace(r"\ ", r"\s+") + r"s?\b"
        if re.search(pattern, text_norm):
            found.append(label)
    return found


def extract_json_object(text):
    text = normalize_text_artifacts(text)
    if not text:
        return None, "empty"
    try:
        return json.loads(text), "ok"
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0)), "recovered_json"
        except json.JSONDecodeError:
            pass
    return None, "parse_error"


def validate_prototype_text(text, class_name=None, min_words=15, max_words=35):
    text = normalize_text_artifacts(text)
    word_count = count_words(text)
    forbidden = find_forbidden_labels(text)
    if class_name:
        class_pattern = r"\b" + re.escape(str(class_name).lower()).replace(r"\ ", r"\s+") + r"s?\b"
        if re.search(class_pattern, text.lower()):
            forbidden.append(str(class_name).lower())
    issues = []
    if not text:
        issues.append("empty")
    if word_count < min_words:
        issues.append("too_short")
    if word_count > max_words:
        issues.append("too_long")
    if forbidden:
        issues.append("contains_forbidden_label")
    return {
        "text": text,
        "word_count": word_count,
        "forbidden_labels": forbidden,
        "status": "ok" if not issues else "warning",
        "issues": issues,
    }


def load_json(path, default):
    if not Path(path).exists():
        return default
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path, payload):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with Path(path).open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def stable_hash(text):
    return hashlib.sha1(normalize_text_artifacts(text).encode("utf-8")).hexdigest()


def l2_normalize(matrix):
    matrix = np.asarray(matrix, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=-1, keepdims=True)
    return matrix / np.clip(norms, 1e-8, None)

## 3. Load And Prepare Descriptions

In [3]:
description_records = load_json(DESCRIPTION_JSON, [])
if isinstance(description_records, dict):
    description_records = list(description_records.values())

train_video_ids = None
train_list_class_counts = Counter()
if USE_VADCLIP_TRAIN_LIST:
    with TRAIN_LIST_CSV.open("r", encoding="utf-8") as f:
        train_rows = list(csv.DictReader(f))
    train_video_ids = {row["video_id"] for row in train_rows}
    video_to_label = {}
    for row in train_rows:
        video_to_label[row["video_id"]] = row["label"]
    train_list_class_counts = Counter(video_to_label.values())
    print("VadCLIP train list unique videos:", len(train_video_ids))
    print("VadCLIP train list class counts:")
    print(json.dumps(dict(sorted(train_list_class_counts.items())), indent=2))

prepared_records = []
skipped_records = []
seen_video_ids = set()
for record in description_records:
    split = record.get("split")
    class_name = record.get("class_name")
    video_id = record.get("video_id")
    description = normalize_text_artifacts(record.get("gpt_description", ""))
    if USE_VADCLIP_TRAIN_LIST:
        if video_id not in train_video_ids:
            continue
    elif split not in SPLITS_TO_USE:
        continue
    if video_id in seen_video_ids:
        skipped_records.append({"video_id": video_id, "class_name": class_name, "reason": "duplicate_video_id"})
        continue
    if not class_name or not video_id:
        skipped_records.append({"video_id": video_id, "reason": "missing_class_or_video_id"})
        continue
    if count_words(description) < MIN_DESCRIPTION_WORDS:
        skipped_records.append({"video_id": video_id, "class_name": class_name, "reason": "description_too_short"})
        continue
    prepared_records.append(
        {
            "split": split,
            "class_name": class_name,
            "video_id": video_id,
            "description": description,
            "description_hash": stable_hash(description),
            "word_count": count_words(description),
        }
    )
    seen_video_ids.add(video_id)

missing_description_video_ids = []
if USE_VADCLIP_TRAIN_LIST:
    missing_description_video_ids = sorted(train_video_ids - seen_video_ids)
    print("VadCLIP train videos without usable GPT description:", len(missing_description_video_ids))
    if missing_description_video_ids:
        print("First missing video_ids:", missing_description_video_ids[:20])

records_by_class = defaultdict(list)
for record in prepared_records:
    records_by_class[record["class_name"]].append(record)

class_counts = {class_name: len(records) for class_name, records in sorted(records_by_class.items())}
print("Prepared records:", len(prepared_records))
print("Skipped records:", len(skipped_records))
print("Classes:", len(class_counts))
print(json.dumps(class_counts, indent=2))

if len(class_counts) != 14:
    raise ValueError(f"Expected 14 classes, found {len(class_counts)}")
if any(count == 0 for count in class_counts.values()):
    raise ValueError("At least one class has no usable descriptions.")

VadCLIP train list unique videos: 1565
VadCLIP train list class counts:
{
  "Abuse": 48,
  "Arrest": 44,
  "Arson": 41,
  "Assault": 45,
  "Burglary": 87,
  "Explosion": 29,
  "Fighting": 45,
  "Normal": 763,
  "RoadAccidents": 125,
  "Robbery": 143,
  "Shooting": 27,
  "Shoplifting": 29,
  "Stealing": 95,
  "Vandalism": 44
}
VadCLIP train videos without usable GPT description: 0
Prepared records: 1565
Skipped records: 0
Classes: 14
{
  "Abuse": 48,
  "Arrest": 44,
  "Arson": 41,
  "Assault": 45,
  "Burglary": 87,
  "Explosion": 29,
  "Fighting": 45,
  "Normal": 763,
  "RoadAccidents": 125,
  "Robbery": 143,
  "Shooting": 27,
  "Shoplifting": 29,
  "Stealing": 95,
  "Vandalism": 44
}


## 4. Embedding And Clustering

In [4]:
def deterministic_dry_embedding(text, dim=256):
    seed = int(stable_hash(text)[:8], 16)
    rng = np.random.default_rng(seed)
    vector = rng.normal(size=dim).astype(np.float32)
    return l2_normalize(vector.reshape(1, -1))[0].tolist()


def call_openai_embeddings(texts):
    if DRY_RUN:
        return [deterministic_dry_embedding(text) for text in texts]
    if not api_key_is_filled(OPENAI_API_KEY):
        raise ValueError("OPENAI_API_KEY is not filled.")
    url = "https://api.openai.com/v1/embeddings"
    headers = {
        "Authorization": f"Bearer {OPENAI_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {"model": EMBEDDING_MODEL, "input": texts}
    response = requests.post(url, headers=headers, json=payload, timeout=REQUEST_TIMEOUT_SECONDS)
    response_json = response.json()
    if response.status_code >= 400:
        raise RuntimeError(f"OpenAI embeddings error {response.status_code}: {json.dumps(response_json)[:1000]}")
    data = sorted(response_json["data"], key=lambda item: item["index"])
    return [item["embedding"] for item in data]


embedding_cache = load_json(EMBEDDING_CACHE_JSON, {})
if embedding_cache.get("embedding_model") != EMBEDDING_MODEL:
    embedding_cache = {"embedding_model": EMBEDDING_MODEL, "items": {}}
embedding_items = embedding_cache.setdefault("items", {})

all_records = prepared_records
missing_records = []
for record in all_records:
    cache_key = f"{record['video_id']}::{record['description_hash']}"
    if cache_key not in embedding_items:
        missing_records.append((cache_key, record))

print("Embedding cache items:", len(embedding_items))
print("Missing embeddings:", len(missing_records))

for start in range(0, len(missing_records), EMBEDDING_BATCH_SIZE):
    batch = missing_records[start:start + EMBEDDING_BATCH_SIZE]
    texts = [record["description"] for _, record in batch]
    embeddings = call_openai_embeddings(texts)
    for (cache_key, record), embedding in zip(batch, embeddings):
        embedding_items[cache_key] = {
            "video_id": record["video_id"],
            "class_name": record["class_name"],
            "description_hash": record["description_hash"],
            "embedding": embedding,
        }
    save_json(EMBEDDING_CACHE_JSON, embedding_cache)
    print(f"Cached embeddings: {min(start + len(batch), len(missing_records))}/{len(missing_records)}")
    if not DRY_RUN and start + EMBEDDING_BATCH_SIZE < len(missing_records):
        time.sleep(SLEEP_BETWEEN_REQUESTS_SECONDS)

print("Embedding cache saved:", EMBEDDING_CACHE_JSON)

Embedding cache items: 0
Missing embeddings: 1565
Cached embeddings: 128/1565
Cached embeddings: 256/1565
Cached embeddings: 384/1565
Cached embeddings: 512/1565
Cached embeddings: 640/1565
Cached embeddings: 768/1565
Cached embeddings: 896/1565
Cached embeddings: 1024/1565
Cached embeddings: 1152/1565
Cached embeddings: 1280/1565
Cached embeddings: 1408/1565
Cached embeddings: 1536/1565
Cached embeddings: 1565/1565
Embedding cache saved: D:\Finetune VadCLIP\code\ucf_class_prototype_embeddings_cache_vadclip_train.json


In [5]:
def simple_numpy_kmeans(embeddings, num_clusters, random_state=42, max_iter=100):
    rng = np.random.default_rng(random_state)
    num_items = embeddings.shape[0]
    initial_indices = rng.choice(num_items, size=num_clusters, replace=False)
    centroids = embeddings[initial_indices].copy()
    labels = np.zeros(num_items, dtype=np.int64)
    for _ in range(max_iter):
        similarities = embeddings @ l2_normalize(centroids).T
        new_labels = np.argmax(similarities, axis=1)
        if np.array_equal(new_labels, labels):
            break
        labels = new_labels
        for cluster_id in range(num_clusters):
            members = embeddings[labels == cluster_id]
            if len(members) == 0:
                centroids[cluster_id] = embeddings[rng.integers(0, num_items)]
            else:
                centroids[cluster_id] = members.mean(axis=0)
    return labels


def choose_num_clusters(class_name, num_descriptions):
    if class_name == "Normal":
        return min(12, max(5, math.ceil(num_descriptions / 60)))
    return min(8, max(2, math.ceil(num_descriptions / 35)))


def cluster_class_records(class_name, records):
    embeddings = []
    for record in records:
        cache_key = f"{record['video_id']}::{record['description_hash']}"
        embeddings.append(embedding_items[cache_key]["embedding"])
    embeddings = l2_normalize(np.asarray(embeddings, dtype=np.float32))

    num_clusters = min(len(records), choose_num_clusters(class_name, len(records)))
    if num_clusters <= 1:
        labels = np.zeros(len(records), dtype=np.int64)
    elif KMeans is None:
        labels = simple_numpy_kmeans(embeddings, num_clusters, random_state=RANDOM_SEED)
    else:
        kmeans = KMeans(n_clusters=num_clusters, random_state=RANDOM_SEED, n_init=10)
        labels = kmeans.fit_predict(embeddings)

    clusters = []
    for cluster_id in range(num_clusters):
        indices = np.where(labels == cluster_id)[0]
        cluster_embeddings = embeddings[indices]
        centroid = l2_normalize(cluster_embeddings.mean(axis=0, keepdims=True))[0]
        similarities = cluster_embeddings @ centroid
        ordered_local = np.argsort(-similarities)
        selected_indices = [int(indices[i]) for i in ordered_local[:MAX_DESCRIPTIONS_PER_CLUSTER_PROMPT]]
        clusters.append(
            {
                "cluster_id": cluster_id,
                "size": int(len(indices)),
                "selected_records": [records[i] for i in selected_indices],
                "selected_video_ids": [records[i]["video_id"] for i in selected_indices],
            }
        )
    return clusters


clusters_by_class = {}
for class_name, records in sorted(records_by_class.items()):
    clusters_by_class[class_name] = cluster_class_records(class_name, records)
    sizes = [cluster["size"] for cluster in clusters_by_class[class_name]]
    print(class_name, "| descriptions:", len(records), "| clusters:", len(sizes), "| sizes:", sizes)


Abuse | descriptions: 48 | clusters: 2 | sizes: [36, 12]
Arrest | descriptions: 44 | clusters: 2 | sizes: [12, 32]
Arson | descriptions: 41 | clusters: 2 | sizes: [16, 25]
Assault | descriptions: 45 | clusters: 2 | sizes: [37, 8]
Burglary | descriptions: 87 | clusters: 3 | sizes: [15, 49, 23]
Explosion | descriptions: 29 | clusters: 2 | sizes: [5, 24]
Fighting | descriptions: 45 | clusters: 2 | sizes: [11, 34]
Normal | descriptions: 763 | clusters: 12 | sizes: [64, 85, 38, 65, 87, 49, 40, 90, 60, 65, 62, 58]
RoadAccidents | descriptions: 125 | clusters: 4 | sizes: [34, 36, 4, 51]
Robbery | descriptions: 143 | clusters: 5 | sizes: [14, 13, 39, 47, 30]
Shooting | descriptions: 27 | clusters: 2 | sizes: [21, 6]
Shoplifting | descriptions: 29 | clusters: 2 | sizes: [27, 2]
Stealing | descriptions: 95 | clusters: 3 | sizes: [21, 20, 54]
Vandalism | descriptions: 44 | clusters: 2 | sizes: [3, 41]


## 5. Candidate Prototype Generation

In [6]:
CANDIDATE_SYSTEM_PROMPT = """You create class-level visual-action prototypes for video anomaly detection.
You must summarize recurring patterns across multiple video descriptions.
Do not copy video-specific details unless they represent a repeated pattern.
Do not mention dataset labels, anomaly category names, or the word anomaly.
Use neutral, visual, action-focused language.
Return only valid JSON."""


def make_numbered_descriptions(records):
    lines = []
    for idx, record in enumerate(records, start=1):
        lines.append(f"{idx}. {record['description']}")
    return "\n".join(lines)


def make_candidate_prompt(class_name, records):
    return f"""Create candidate class-level prototypes from the following video descriptions.

Class name for internal reference only: {class_name}
Do not mention this class name in the output.

Requirements:
- Produce exactly {CANDIDATES_PER_CLUSTER} candidate prototypes.
- Each prototype must be 15 to 35 words.
- Focus on recurring actors, visible actions, interactions, objects, and temporal patterns.
- Ignore clothing colors, camera viewpoint, exact scene layout, and one-off background details.
- Do not use these forbidden words: {', '.join(FORBIDDEN_LABELS)}.
- Do not invent patterns that are not supported by the descriptions.
- Return exactly this JSON shape:
{{
  "candidate_prototypes": [
    {{"text": "...", "rationale": "..."}}
  ]
}}

Video descriptions:
{make_numbered_descriptions(records)}"""


def extract_openai_output_text(response_json):
    if response_json.get("output_text"):
        return normalize_text_artifacts(response_json["output_text"])
    chunks = []

    def visit(value):
        if isinstance(value, dict):
            value_type = value.get("type")
            if value_type in {"output_text", "text"} and isinstance(value.get("text"), str):
                chunks.append(value["text"])
            if isinstance(value.get("content"), str):
                chunks.append(value["content"])
            for child in value.values():
                visit(child)
        elif isinstance(value, list):
            for child in value:
                visit(child)

    visit(response_json.get("output", []))
    return normalize_text_artifacts("\n".join(chunk for chunk in chunks if chunk).strip())


def call_openai_response(system_prompt, user_prompt):
    if DRY_RUN:
        return "", {"skipped": True, "reason": "DRY_RUN is True"}
    if not api_key_is_filled(OPENAI_API_KEY):
        raise ValueError("OPENAI_API_KEY is not filled.")
    url = "https://api.openai.com/v1/responses"
    headers = {
        "Authorization": f"Bearer {OPENAI_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": GENERATION_MODEL,
        "input": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "max_output_tokens": GENERATION_MAX_OUTPUT_TOKENS,
    }
    response = requests.post(url, headers=headers, json=payload, timeout=REQUEST_TIMEOUT_SECONDS)
    response_json = response.json()
    if response.status_code >= 400:
        raise RuntimeError(f"OpenAI responses error {response.status_code}: {json.dumps(response_json)[:1000]}")
    return extract_openai_output_text(response_json), response_json


def dry_candidate_payload(class_name, cluster_id):
    return {
        "candidate_prototypes": [
            {
                "text": "People move through the scene with repeated visible interactions, object handling, and changing body positions that form a shared action pattern.",
                "rationale": "Dry-run placeholder for candidate prototype generation.",
            },
            {
                "text": "Individuals approach, separate, or gather around objects and other people while the scene develops through several connected actions.",
                "rationale": "Dry-run placeholder for candidate prototype generation.",
            },
            {
                "text": "The visual sequence emphasizes body movement, nearby objects, and interactions that repeat across multiple descriptions in this cluster.",
                "rationale": "Dry-run placeholder for candidate prototype generation.",
            },
        ]
    }


def generate_candidate_prototypes(class_name, cluster):
    if DRY_RUN:
        payload = dry_candidate_payload(class_name, cluster["cluster_id"])
        return payload, {"parse_status": "dry_run", "attempts": []}

    prompt = make_candidate_prompt(class_name, cluster["selected_records"])
    attempts = []
    last_payload = None
    last_parse_status = "not_started"
    for attempt_idx in range(1, MAX_RETRIES + 1):
        try:
            raw_text, response_json = call_openai_response(CANDIDATE_SYSTEM_PROMPT, prompt)
            payload, parse_status = extract_json_object(raw_text)
            attempts.append({"attempt": attempt_idx, "parse_status": parse_status})
            last_payload = payload
            last_parse_status = parse_status
            candidates = payload.get("candidate_prototypes", []) if isinstance(payload, dict) else []
            if len(candidates) == CANDIDATES_PER_CLUSTER:
                return payload, {"parse_status": parse_status, "attempts": attempts}
        except Exception as exc:
            attempts.append({"attempt": attempt_idx, "error": repr(exc)})
        if attempt_idx < MAX_RETRIES:
            time.sleep(RETRY_BACKOFF_SECONDS * attempt_idx)
    return last_payload or {"candidate_prototypes": []}, {"parse_status": last_parse_status, "attempts": attempts}


In [7]:
candidate_results_by_class = defaultdict(list)
candidate_generation_report = []

for class_name, clusters in sorted(clusters_by_class.items()):
    print("Generating candidates for class:", class_name)
    for cluster in clusters:
        payload, debug = generate_candidate_prototypes(class_name, cluster)
        raw_candidates = payload.get("candidate_prototypes", []) if isinstance(payload, dict) else []
        for idx, candidate in enumerate(raw_candidates[:CANDIDATES_PER_CLUSTER]):
            validation = validate_prototype_text(candidate.get("text", ""), class_name=class_name)
            candidate_record = {
                "class_name": class_name,
                "cluster_id": cluster["cluster_id"],
                "cluster_size": cluster["size"],
                "candidate_index": idx,
                "text": validation["text"],
                "rationale": normalize_text_artifacts(candidate.get("rationale", "")),
                "validation": validation,
                "selected_video_ids": cluster["selected_video_ids"],
            }
            candidate_results_by_class[class_name].append(candidate_record)
        candidate_generation_report.append(
            {
                "class_name": class_name,
                "cluster_id": cluster["cluster_id"],
                "cluster_size": cluster["size"],
                "selected_descriptions": len(cluster["selected_records"]),
                "num_candidates": len(raw_candidates),
                "debug": debug,
            }
        )
        if not DRY_RUN:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SECONDS)

for class_name, candidates in sorted(candidate_results_by_class.items()):
    statuses = Counter(candidate["validation"]["status"] for candidate in candidates)
    print(class_name, "| candidates:", len(candidates), "| statuses:", dict(statuses))

Generating candidates for class: Abuse
Generating candidates for class: Arrest
Generating candidates for class: Arson
Generating candidates for class: Assault
Generating candidates for class: Burglary
Generating candidates for class: Explosion
Generating candidates for class: Fighting
Generating candidates for class: Normal
Generating candidates for class: RoadAccidents
Generating candidates for class: Robbery
Generating candidates for class: Shooting
Generating candidates for class: Shoplifting
Generating candidates for class: Stealing
Generating candidates for class: Vandalism
Abuse | candidates: 6 | statuses: {'ok': 6}
Arrest | candidates: 6 | statuses: {'ok': 6}
Arson | candidates: 6 | statuses: {'ok': 6}
Assault | candidates: 6 | statuses: {'ok': 6}
Burglary | candidates: 9 | statuses: {'ok': 9}
Explosion | candidates: 6 | statuses: {'ok': 6}
Fighting | candidates: 6 | statuses: {'ok': 6}
Normal | candidates: 36 | statuses: {'ok': 36}
RoadAccidents | candidates: 12 | statuses: {'o

## 6. Final Prototype Synthesis

In [8]:
FINAL_SYSTEM_PROMPT = """You synthesize final class-level visual-action prototypes for video anomaly detection.
You must merge overlapping candidate patterns and preserve only general, recurring visual semantics.
Do not mention dataset labels, anomaly category names, or the word anomaly.
Return only valid JSON."""


def make_candidate_prototypes_text(candidates):
    lines = []
    for idx, candidate in enumerate(candidates, start=1):
        lines.append(f"{idx}. {candidate['text']}")
    return "\n".join(lines)


def make_final_prompt(class_name, candidates):
    return f"""Synthesize final class-level prototypes from candidate prototypes.

Class name for internal reference only: {class_name}
Do not mention this class name in the output.

Requirements:
- Produce exactly {FINAL_PROTOTYPES_PER_CLASS} final prototypes.
- Use these prototype types exactly:
  core_action
  actor_object_interaction
  temporal_progression
  scene_context
  hard_negative_distinction
- Each prototype must be 15 to 35 words.
- Keep prototypes general enough to represent the class, not a single video.
- Prefer visual actions and interactions over scene-specific details.
- Do not use these forbidden words: {', '.join(FORBIDDEN_LABELS)}.
- For Normal, describe routine behavior, ordinary interactions, and absence of visible disruption without using anomaly-related words.
- Return exactly this JSON shape:
{{
  "final_prototypes": [
    {{"type": "core_action", "text": "..."}},
    {{"type": "actor_object_interaction", "text": "..."}},
    {{"type": "temporal_progression", "text": "..."}},
    {{"type": "scene_context", "text": "..."}},
    {{"type": "hard_negative_distinction", "text": "..."}}
  ]
}}

Candidate prototypes:
{make_candidate_prototypes_text(candidates)}"""


def dry_final_payload(class_name):
    return {
        "final_prototypes": [
            {"type": "core_action", "text": "People perform repeated visible actions that define the main event pattern across multiple related surveillance videos."},
            {"type": "actor_object_interaction", "text": "Individuals interact with nearby people or objects in ways that repeatedly shape the central visual sequence."},
            {"type": "temporal_progression", "text": "The scene develops from ordinary movement into a more focused sequence of actions, reactions, and departures."},
            {"type": "scene_context", "text": "Events commonly occur in public, indoor, roadside, or commercial areas where people and objects remain clearly visible."},
            {"type": "hard_negative_distinction", "text": "The pattern should be separated from routine motion by emphasizing repeated purposeful interactions and visible scene changes."},
        ]
    }


def generate_final_prototypes(class_name, candidates):
    if DRY_RUN:
        payload = dry_final_payload(class_name)
        return payload, {"parse_status": "dry_run", "attempts": []}

    prompt = make_final_prompt(class_name, candidates)
    attempts = []
    last_payload = None
    last_parse_status = "not_started"
    for attempt_idx in range(1, MAX_RETRIES + 1):
        try:
            raw_text, response_json = call_openai_response(FINAL_SYSTEM_PROMPT, prompt)
            payload, parse_status = extract_json_object(raw_text)
            attempts.append({"attempt": attempt_idx, "parse_status": parse_status})
            last_payload = payload
            last_parse_status = parse_status
            prototypes = payload.get("final_prototypes", []) if isinstance(payload, dict) else []
            types = [prototype.get("type") for prototype in prototypes]
            if len(prototypes) == FINAL_PROTOTYPES_PER_CLASS and types == PROTOTYPE_TYPES:
                return payload, {"parse_status": parse_status, "attempts": attempts}
        except Exception as exc:
            attempts.append({"attempt": attempt_idx, "error": repr(exc)})
        if attempt_idx < MAX_RETRIES:
            time.sleep(RETRY_BACKOFF_SECONDS * attempt_idx)
    return last_payload or {"final_prototypes": []}, {"parse_status": last_parse_status, "attempts": attempts}


In [9]:
final_results_by_class = {}
final_generation_report = []

for class_name, candidates in sorted(candidate_results_by_class.items()):
    print("Synthesizing final prototypes for class:", class_name)
    payload, debug = generate_final_prototypes(class_name, candidates)
    raw_prototypes = payload.get("final_prototypes", []) if isinstance(payload, dict) else []
    final_prototypes = []
    for prototype in raw_prototypes:
        validation = validate_prototype_text(prototype.get("text", ""), class_name=class_name)
        final_prototypes.append(
            {
                "type": prototype.get("type", ""),
                "text": validation["text"],
                "validation": validation,
            }
        )
    final_results_by_class[class_name] = final_prototypes
    final_generation_report.append(
        {
            "class_name": class_name,
            "num_candidates": len(candidates),
            "num_final_prototypes": len(final_prototypes),
            "debug": debug,
        }
    )
    if not DRY_RUN:
        time.sleep(SLEEP_BETWEEN_REQUESTS_SECONDS)

for class_name, prototypes in sorted(final_results_by_class.items()):
    statuses = Counter(prototype["validation"]["status"] for prototype in prototypes)
    print(class_name, "| final prototypes:", len(prototypes), "| statuses:", dict(statuses))

Synthesizing final prototypes for class: Abuse
Synthesizing final prototypes for class: Arrest
Synthesizing final prototypes for class: Arson
Synthesizing final prototypes for class: Assault
Synthesizing final prototypes for class: Burglary
Synthesizing final prototypes for class: Explosion
Synthesizing final prototypes for class: Fighting
Synthesizing final prototypes for class: Normal
Synthesizing final prototypes for class: RoadAccidents
Synthesizing final prototypes for class: Robbery
Synthesizing final prototypes for class: Shooting
Synthesizing final prototypes for class: Shoplifting
Synthesizing final prototypes for class: Stealing
Synthesizing final prototypes for class: Vandalism
Abuse | final prototypes: 5 | statuses: {'ok': 5}
Arrest | final prototypes: 5 | statuses: {'ok': 5}
Arson | final prototypes: 5 | statuses: {'ok': 5}
Assault | final prototypes: 5 | statuses: {'ok': 5}
Burglary | final prototypes: 5 | statuses: {'ok': 5}
Explosion | final prototypes: 5 | statuses: {'

## 7. Validate And Save

In [10]:
validation_errors = []
for class_name in sorted(records_by_class):
    prototypes = final_results_by_class.get(class_name, [])
    if len(prototypes) != FINAL_PROTOTYPES_PER_CLASS:
        validation_errors.append(f"{class_name}: expected {FINAL_PROTOTYPES_PER_CLASS} final prototypes, got {len(prototypes)}")
        continue
    prototype_types = [prototype["type"] for prototype in prototypes]
    if prototype_types != PROTOTYPE_TYPES:
        validation_errors.append(f"{class_name}: prototype types mismatch: {prototype_types}")
    for prototype in prototypes:
        if prototype["validation"]["status"] != "ok":
            validation_errors.append(f"{class_name}/{prototype['type']}: {prototype['validation']['issues']}")

if validation_errors:
    print("Validation warnings/errors:")
    for error in validation_errors[:50]:
        print("-", error)
else:
    print("All final prototypes passed validation.")

output_payload = {
    "metadata": {
        "source": "GPT video descriptions from UCA timestamp annotations",
        "selection_mode": "vadclip_train_list" if USE_VADCLIP_TRAIN_LIST else "uca_split",
        "splits_used": SPLITS_TO_USE,
        "train_list_csv": str(TRAIN_LIST_CSV) if USE_VADCLIP_TRAIN_LIST else None,
        "embedding_model": EMBEDDING_MODEL,
        "generation_model": GENERATION_MODEL,
        "final_prototypes_per_class": FINAL_PROTOTYPES_PER_CLASS,
        "candidates_per_cluster": CANDIDATES_PER_CLUSTER,
        "max_descriptions_per_cluster_prompt": MAX_DESCRIPTIONS_PER_CLUSTER_PROMPT,
        "dry_run": DRY_RUN,
    },
    "classes": {},
}

for class_name in sorted(records_by_class):
    output_payload["classes"][class_name] = {
        "num_descriptions": len(records_by_class[class_name]),
        "num_clusters": len(clusters_by_class[class_name]),
        "final_prototypes": [
            {"type": prototype["type"], "text": prototype["text"]}
            for prototype in final_results_by_class.get(class_name, [])
        ],
        "candidate_prototypes": [
            {
                "cluster_id": candidate["cluster_id"],
                "text": candidate["text"],
                "rationale": candidate["rationale"],
            }
            for candidate in candidate_results_by_class.get(class_name, [])
        ],
    }

report_payload = {
    "metadata": output_payload["metadata"],
    "class_counts": class_counts,
    "train_list_class_counts": dict(sorted(train_list_class_counts.items())) if USE_VADCLIP_TRAIN_LIST else {},
    "missing_description_video_ids": missing_description_video_ids,
    "skipped_records": skipped_records,
    "candidate_generation_report": candidate_generation_report,
    "final_generation_report": final_generation_report,
    "validation_errors": validation_errors,
}

save_json(OUTPUT_JSON, output_payload)
save_json(REPORT_JSON, report_payload)

with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as f:
    fieldnames = ["class_name", "prototype_type", "prototype_text", "num_descriptions", "num_clusters"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for class_name, class_payload in output_payload["classes"].items():
        for prototype in class_payload["final_prototypes"]:
            writer.writerow(
                {
                    "class_name": class_name,
                    "prototype_type": prototype["type"],
                    "prototype_text": prototype["text"],
                    "num_descriptions": class_payload["num_descriptions"],
                    "num_clusters": class_payload["num_clusters"],
                }
            )

print("Saved JSON:", OUTPUT_JSON)
print("Saved CSV:", OUTPUT_CSV)
print("Saved report:", REPORT_JSON)

All final prototypes passed validation.
Saved JSON: D:\Finetune VadCLIP\code\ucf_class_description_prototypes_vadclip_train.json
Saved CSV: D:\Finetune VadCLIP\code\ucf_class_description_prototypes_vadclip_train.csv
Saved report: D:\Finetune VadCLIP\code\ucf_class_prototype_generation_report_vadclip_train.json


## 8. Preview

In [11]:
for class_name in ["Normal", "Robbery", "Shoplifting", "Fighting", "Assault"]:
    if class_name not in output_payload["classes"]:
        continue
    print("\n" + class_name)
    for prototype in output_payload["classes"][class_name]["final_prototypes"]:
        print(f"- {prototype['type']}: {prototype['text']}")


Normal
- core_action: People and vehicles follow ordinary routes, repeatedly walking, driving, stopping, turning, entering, exiting, and continuing through shared spaces.
- actor_object_interaction: Individuals routinely handle documents, bags, merchandise, tools, and vehicles, exchanging objects, operating counters, opening doors, and organizing nearby areas.
- temporal_progression: Activity unfolds through repeated cycles of arrival, brief pauses, conversation, transactions, movement between areas, and departure, with different participants replacing one another over time.
- scene_context: Shared roads, workplaces, stores, corridors, dining areas, and public interiors show steady circulation, organized routines, and people using spaces for everyday purposes.
- hard_negative_distinction: Brief stops, queues, gatherings, and vehicle maneuvering remain coordinated and purposeful, with participants resuming ordinary movement rather than causing visible disruption.

Robbery
- core_action